## setup

In [ ]:
from typing import Any, Optional, List, Literal, Dict, Union, Tuple, Iterator
from pydantic import BaseModel, Field, PrivateAttr
from enum import Enum
from collections import defaultdict

import json, time, requests, re, random

from uuid import uuid4
from datetime import datetime
from pathlib import Path
from decouple import config
from tqdm import tqdm
from urllib.parse import unquote
from statistics import mean, quantiles
from PIL import Image

In [ ]:
DATA_DIR = "data"

SOEUR_BASE_URL = "https://secondhand.soeur.fr/products"
VINTED_BASE_URL = "https://www.vinted.fr"
VINTED_API_URL = f"{VINTED_BASE_URL}/api/v2"
VINTED_BRAND_ID = 18751

### models

In [41]:
class Item(BaseModel):
    id: str
    slug: str
    title: str
    min_price: int
    max_price: int
    origin_price: int
    image_url: str
    state: Optional[str] = None
    size: Optional[str] = None
    type: Optional[str] = None
    year: Optional[str] = None
    color: Optional[str] = None
    gender: Optional[str] = None
    modele: Optional[str] = None
    season: Optional[str] = None
    material: Optional[str] = None
    description: Optional[str] = None

In [42]:
class VintedEndpoint(Enum):
    CATALOG_ITEMS = "/catalog/items"
    CATALOG_INITIALIZERS = "/catalog/initializers"
    CATALOG_FILTERS = "/catalog/filters"
    CATALOG_FILTERS_SEARCH = "/catalog/filters/search"
    ITEMS = "/items/{}"
    USERS = "/users"
    USER = "/users/{}"
    USER_FEEDBACKS = "/user_feedbacks"
    USER_ITEMS = "/wardrobe/{}/items"
    USER_FEEDBACKS_SUMMARY = "/user_feedbacks/summary"
    SEARCH_SUGGESTIONS = "/search_suggestions"


VintedSortOption = Literal[
    "relevance", "price_high_to_low", "price_low_to_high", "newest_first"
]


class VintedSearchParams(BaseModel):
    brand_ids: List[int]
    page: int = 1
    query: Optional[str] = None
    catalog_ids: List[int] = Field(default_factory=list)
    color_ids: List[int] = Field(default_factory=list)
    material_ids: List[int] = Field(default_factory=list)
    sort_option: VintedSortOption = "newest_first"
    

In [43]:
class ProxyConfig(BaseModel):
    password: str = config("APIFY_PROXY_PASSWORD")
    country_code: str = "FR"
    _hostname: str = "proxy.apify.com"
    _port: int = 8000

    @property
    def url_datacenter(self) -> str:
        username = f"auto:{self.password}"
        return f"http://{username}@{self._hostname}:{self._port}"

    @property
    def url_residential(self) -> str:
        username = f"groups-RESIDENTIAL,country-{self.country_code}:{self.password}"
        return f"http://{username}@{self._hostname}:{self._port}"

    @property
    def url(self) -> str:
        return self.url_residential

In [44]:
class VintedResponse(BaseModel):
    status_code: int = 500
    data: Optional[Dict] = None
    error: Optional[str] = None

    @property
    def ok(self) -> bool:
        return self.status_code == 200

    @property
    def items(self) -> Optional[List[Dict]]:
        return self.data.get("items")

    @property
    def pagination(self) -> Optional[Dict]:
        if self.data is None:
            return None

        return self.data.get("pagination")

    @property
    def total_pages(self) -> Optional[int]:
        pagination = self.pagination
        if pagination:
            return pagination.get("total_pages")

        return None

    @property
    def current_page(self) -> Optional[int]:
        pagination = self.pagination
        if pagination:
            return pagination.get("current_page")

        return None

    @property
    def total_entries(self) -> Optional[int]:
        pagination = self.pagination
        if pagination:
            return pagination.get("total_entries")

        return None

In [45]:
class VintedItem(BaseModel):
    id: int
    url: str
    image_url: str
    title: str
    brand: str
    size: str
    status: str
    price: int  

In [70]:
class DatasetEntry(BaseModel):
    item: Item
    vinted_items: List[VintedItem]

    def to_json(self) -> Dict[str, Any]:
        return {
            "item": self.item.model_dump(),
            "vinted_items": [vinted_item.model_dump() for vinted_item in self.vinted_items],
        }


class Dataset(BaseModel):
    entries: List[DatasetEntry] = Field(default_factory=list)
    _index: set[str] = PrivateAttr(default_factory=set)

    def model_post_init(self, __context: Any) -> None:
        self._index = {entry.item.id for entry in self.entries}

    def __len__(self) -> int:
        return len(self.entries)

    def __getitem__(self, idx: int) -> DatasetEntry:
        return self.entries[idx]

    def __iter__(self) -> Iterator[DatasetEntry]:
        return iter(self.entries)

    def add(self, item: Item, vinted_items: List[VintedItem]) -> None:
        if item.id in self._index:
            return

        entry = DatasetEntry(item=item, vinted_items=vinted_items)
        self.entries.append(entry)
        self._index.add(item.id)
    
    def to_json(self) -> List[Dict[str, Any]]:
        return [entry.to_json() for entry in self.entries]

    @classmethod
    def from_json(cls, json_data: List[Dict[str, Any]]) -> "Dataset":
        dataset = cls()
        
        for entry in json_data:
            item = Item(**entry["item"])
            vinted_items = [VintedItem(**data) for data in entry["vinted_items"]]
            dataset.add(item, vinted_items)

        return dataset

In [ ]:
class VintedComparablesEntry(BaseModel):
    faume_item_id: str
    faume_slug: Optional[str] = None
    faume_title: Optional[str] = None
    faume_image_url: Optional[str] = None
    faume_min_price: Optional[int] = None
    faume_max_price: Optional[int] = None
    faume_origin_price: Optional[int] = None
    faume_display_price: Optional[float] = None
    faume_state: Optional[str] = None
    faume_size: Optional[str] = None
    faume_type: Optional[str] = None
    faume_year: Optional[str] = None
    faume_color: Optional[str] = None
    faume_gender: Optional[str] = None
    faume_modele: Optional[str] = None
    faume_season: Optional[str] = None
    faume_material: Optional[str] = None
    faume_description: Optional[str] = None

    comparables_count: Optional[int] = None
    comparables_count_bucket: Optional[str] = None

    vinted_item_id: int
    vinted_url: Optional[str] = None
    vinted_image_url: Optional[str] = None
    vinted_title: Optional[str] = None
    vinted_brand: Optional[str] = None
    vinted_size: Optional[str] = None
    vinted_status: Optional[str] = None
    vinted_price: Optional[int] = None

    price_diff_abs: Optional[float] = None
    price_diff_rel: Optional[float] = None

    created_at: Optional[datetime] = None
    run_id: Optional[str] = None

### functions

In [101]:
def load_json(path: str) -> Optional[Any]:
    try:
        with open(path, "r") as f:
            data = json.load(f)

        return data

    except FileNotFoundError:
        return None

    
def load_json_lines(path: str) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def save_json(data: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)


def save_json_lines(data: List[Dict[str, Any]], path: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [51]:
def get_first_choice(data: Dict[str, Any]) -> Dict[str, Any]:
    choices = data.get("choices") or []

    return choices[0] if isinstance(choices, List) and choices else {}


def int_or_zero(v: Any) -> int:
    if v is None or v == "":
        return 0
    try:
        return int(v)
    except (TypeError, ValueError):
        return 0


def get_first_photo(data: Dict[str, Any]) -> Optional[str]:
    photos = data.get("photos")
    if isinstance(photos, List) and photos:
        first = photos[0]
        return first if isinstance(first, str) else None
    return None


def extract_id(data: Dict[str, Any]) -> str:
    return str(data.get("id") or "")


def extract_slug(data: Dict[str, Any]) -> str:
    return str(data.get("slug") or "")


def extract_title(data: Dict[str, Any]) -> str:
    return str(data.get("title") or data.get("title") or "")


def extract_min_price(data: Dict[str, Any]) -> int:
    return int_or_zero(data.get("minPrice"))


def extract_max_price(data: Dict[str, Any]) -> int:
    return int_or_zero(data.get("maxPrice"))


def extract_origin_price(data: Dict[str, Any]) -> int:
    return int_or_zero(data.get("originPrice"))


def extract_image_url(data: Dict[str, Any]) -> str:
    return (
        data.get("frontPhoto")
        or data.get("backPhoto")
        or get_first_photo(data)
        or ""
    )


def extract_state(data: Dict[str, Any]) -> Optional[str]:
    return data.get("state")


def extract_size(data: Dict[str, Any]) -> Optional[str]:
    return data.get("size")


def extract_type(data: Dict[str, Any]) -> Optional[str]:
    return data.get("type")


def extract_year(data: Dict[str, Any]) -> Optional[str]:
    return data.get("year")


def extract_color(data: Dict[str, Any]) -> Optional[str]:
    raw = data.get("color") or data.get("color")
    if raw is None:
        return None
    s = str(raw).strip()
    return s or None


def extract_gender(data: Dict[str, Any]) -> Optional[str]:
    return data.get("gender")


def extract_modele(data: Dict[str, Any]) -> Optional[str]:
    return data.get("modele")


def extract_season(data: Dict[str, Any]) -> Optional[str]:
    return data.get("season")


def extract_material(data: Dict[str, Any]) -> Optional[str]:
    return data.get("material")


def extract_description(data: Dict[str, Any]) -> Optional[str]:
    return data.get("description")


def convert_to_item(item_json: Dict[str, Any]) -> Item:
    first_choice = get_first_choice(item_json)

    return Item(
        id=extract_id(item_json),
        slug=extract_slug(item_json),
        title=extract_title(item_json),
        min_price=extract_min_price(item_json),
        max_price=extract_max_price(item_json),
        origin_price=extract_origin_price(item_json),
        image_url=extract_image_url(item_json),
        state=extract_state(first_choice),
        size=extract_size(first_choice),
        type=extract_type(first_choice),
        year=extract_year(first_choice),
        color=extract_color(first_choice),
        gender=extract_gender(first_choice),
        modele=extract_modele(first_choice),
        season=extract_season(first_choice),
        material=extract_material(first_choice),
        description=extract_description(first_choice),
    )

In [52]:
def extract_vinted_id(data: Dict[str, Any]) -> int:
    return data["id"]


def extract_vinted_url(data: Dict[str, Any]) -> str:
    return data["url"]


def extract_vinted_image_url(data: Dict[str, Any]) -> str:
    return data["photo"]["url"]


def extract_vinted_title(data: Dict[str, Any]) -> str:
    return data["title"]


def extract_vinted_brand(data: Dict[str, Any]) -> str:
    return data["brand_title"]


def extract_vinted_size(data: Dict[str, Any]) -> str:
    return data["size_title"]


def extract_vinted_status(data: Dict[str, Any]) -> str:
    return data["status"]


def extract_vinted_price(data: Dict[str, Any]) -> int:
    price_amount = float(data["price"]["amount"])
    price_amount = int(price_amount * 100)      

    return price_amount


def extract_vinted_status(data: Dict[str, Any]) -> str:
    return data["status"]


def convert_to_vinted_item(item_json: Dict[str, Any]) -> VintedItem:
    return VintedItem(
        id=extract_vinted_id(item_json),
        url=extract_vinted_url(item_json),
        image_url=extract_vinted_image_url(item_json),
        title=extract_vinted_title(item_json),
        brand=extract_vinted_brand(item_json),
        size=extract_vinted_size(item_json),
        status=extract_vinted_status(item_json),
        price=extract_vinted_price(item_json),
    )

In [53]:
def get_statistics(
    values: List[Union[int, float]], quantiles_n: int = 4 
) -> Dict:
    if values is None or len(values) == 0:
        raise ValueError("values must be a non-empty List of numbers")

    result = {
        "average": mean(values),
        "min": min(values),
        "max": max(values)
    }

    quantiles_values = quantiles(values, n=quantiles_n, method="inclusive")
 
    for i, quantile in enumerate(quantiles_values):
        result[f"q{i+1}"] = quantile

    return result


def calculate_difference(x: Union[int, float], y: Union[int, float], relative: bool = False) -> float:
    diff = y - x

    if relative:
        return diff / x

    return diff

In [132]:
def get_comparables_count_bucket(
    comparables_count: int,
    mapping: Dict[int, str],
) -> str:
    max_threshold = -1
    value = mapping.get(max_threshold)
    if not value:
        raise ValueError("No default bucket found")

    for threshold, bucket in mapping.items():
        if comparables_count >= threshold and threshold > max_threshold:
            value = bucket
            max_threshold = threshold

    return value


def build_vinted_comparables_entries(
    dataset_entry: DatasetEntry, 
    comparables_count_bucket_mapping: Dict[int, str],
    run_id: Optional[str] = None, 
    created_at: Optional[datetime] = None,
    as_json: bool = False,
) -> List[Union[VintedComparablesEntry, Dict[str, Any]]]:
    entries = []

    comparables_count = len(dataset_entry.vinted_items)
    
    comparables_count_bucket = get_comparables_count_bucket(
        comparables_count=comparables_count,
        mapping=comparables_count_bucket_mapping,
    )

    for vinted_item in dataset_entry.vinted_items:
        item = dataset_entry.item
        x, y = vinted_item.price, item.max_price
        price_diff_abs = calculate_difference(x, y, relative=False)
        price_diff_rel = calculate_difference(x, y, relative=True)

        entry = VintedComparablesEntry(
            faume_item_id=item.id,  
            faume_slug=item.slug,
            faume_title=item.title,
            faume_image_url=item.image_url,
            faume_min_price=item.min_price,
            faume_max_price=item.max_price,
            faume_origin_price=item.origin_price,
            faume_display_price=item.max_price,
            faume_state=item.state,
            faume_size=item.size,
            faume_type=item.type,
            faume_year=item.year,
            faume_color=item.color,
            faume_gender=item.gender,
            faume_modele=item.modele,
            faume_season=item.season,
            faume_material=item.material,
            faume_description=item.description,
            comparables_count=comparables_count,
            comparables_count_bucket=comparables_count_bucket,
            vinted_item_id=vinted_item.id,
            vinted_url=vinted_item.url,
            vinted_image_url=vinted_item.image_url,
            vinted_title=vinted_item.title,
            vinted_brand=vinted_item.brand,
            vinted_size=vinted_item.size,
            vinted_status=vinted_item.status,
            vinted_price=vinted_item.price,
            price_diff_abs=price_diff_abs,
            price_diff_rel=price_diff_rel,
            created_at=created_at,
            run_id=run_id,
        )

        if as_json:
            entry = entry.model_dump(mode="json")

        entries.append(entry)

    return entries

### `VintedClient`

In [55]:
def parse_url_to_params(url: str):
    try:
        decoded_url = unquote(url)

        matched_params = re.match(r"^https:\/\/www\.vinted\.([a-z]+)", decoded_url)
        if not matched_params:
            raise Exception("Invalid URL")

        missing_ids_params = ["catalog", "status"]

        params = re.findall(r"([a-z_]+)(\[\])?=([a-zA-Z 0-9._À-ú+%]*)&?", decoded_url)
        if not isinstance(matched_params.groups(), tuple):
            raise Exception("Invalid URL")

        mapped_params = {}

        for param_name, is_array, param_value in params:
            if " " in param_value:
                param_value = param_value.replace(" ", "+")

            if is_array:
                if param_name in missing_ids_params:
                    param_name = f"{param_name}_id"

                if param_name + "s" in mapped_params:
                    mapped_params[param_name + "s"].append(param_value)
                else:
                    mapped_params[param_name + "s"] = [param_value]
            else:
                mapped_params[param_name] = param_value

        final_params = {}
        for key, value in mapped_params.items():
            if isinstance(value, List):
                final_params[key] = ",".join(value)
            else:
                final_params[key] = value

        [
            final_params.pop(key)
            for key in ["time", "page", "per_page"]
            if key in final_params.keys()
        ]

        return final_params

    except Exception as e:
        raise Exception("Invalid URL")

In [56]:
class VintedClient:
    USER_AGENT = (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    BASE_HEADERS = {
        "Accept": "application/json",
        "Accept-Language": "en-US,en;q=0.9",
        "User-Agent": USER_AGENT,
    }

    def __init__(self, proxy_config: Optional[ProxyConfig] = None) -> None:
        self.cookies = None
        self._use_proxy = False
        self.proxy_config = proxy_config

        self.REQUESTS_KWARGS = {
            "headers": {**self.BASE_HEADERS, "Referer": VINTED_BASE_URL},
            "allow_redirects": True,
            "timeout": 30,
        }

    def fetch_cookies(self):
        response = self._call(method="get", url=VINTED_BASE_URL)
        self.cookies = response.cookies

    def search(
        self,
        page: int = 1,
        per_page: int = 96,
        url: Optional[str] = None,
        query: Optional[str] = None,
        price_from: Optional[float] = None,
        price_to: Optional[float] = None,
        sort_option: VintedSortOption = "newest_first",
        catalog_ids: Optional[List[int]] = None,
        size_ids: Optional[List[int]] = None,
        brand_ids: Optional[List[int]] = None,
        status_ids: Optional[List[int]] = None,
        color_ids: Optional[List[int]] = None,
        patterns_ids: Optional[List[int]] = None,
        material_ids: Optional[List[int]] = None,
    ) -> VintedResponse:
        params = {
            "page": page,
            "per_page": per_page,
            "time": time.time(),
            "search_text": query,
            "price_from": price_from,
            "price_to": price_to,
            "catalog_ids": catalog_ids,
            "order": sort_option,
            "size_ids": size_ids,
            "brand_ids": brand_ids,
            "status_ids": status_ids,
            "color_ids": color_ids,
            "patterns_ids": patterns_ids,
            "material_ids": material_ids,
        }
        if url:
            params.update(parse_url_to_params(url))

        return self._get(VintedEndpoint.CATALOG_ITEMS, params=params)

    def _get(
        self,
        endpoint: VintedEndpoint,
        format_values: Optional[Dict] = None,
        *args,
        **kwargs,
    ) -> VintedResponse:
        if format_values:
            url = VINTED_API_URL + endpoint.value.format(format_values)
        else:
            url = VINTED_API_URL + endpoint.value

        try:
            response = self._call(method="get", url=url, *args, **kwargs)

            if response.status_code == 200:
                model = VintedResponse(
                    status_code=response.status_code, data=response.json()
                )
            else:
                model = VintedResponse(
                    status_code=response.status_code, error=response.text
                )

        except Exception as e:
            model = VintedResponse(status_code=500, error=str(e))

        if not model.ok and not self._use_proxy and self.proxy_config:
            self._use_proxy = True
            return self._get(endpoint, format_values, *args, **kwargs)

        return model

    def _call(self, method: Literal["get"], *args, **kwargs):
        kwargs.update(self.REQUESTS_KWARGS)
        use_proxy = self._use_proxy or not self.cookies

        if self.cookies:
            kwargs["cookies"] = self.cookies

        if self.proxy_config and use_proxy:
            kwargs["proxies"] = {
                "http": self.proxy_config.url,
                "https": self.proxy_config.url,
            }

        return requests.request(method=method, *args, **kwargs)

## data

### load

In [57]:
data = load_json(f"{DATA_DIR}/items.json")
len(data)

1369

In [58]:
items = [convert_to_item(item_json) for item_json in data]
len(items)

1369

In [59]:
items[0]

Item(id='019b35e2-068e-7b45-ac15-7ff7c77ca67e', slug='baden-shirt-CHE1536BADEN-absinth', title='BADEN SHIRT', min_price=19500, max_price=19500, origin_price=32500, image_url='https://imagedelivery.net/pSib6VZeYacCoggJMcMUVw/8208a6fc7f80fce0661c18f80bea70a9_19966_694323fa1be9b/public', state='Excellent condition', size='36', type='SHIRT', year='2024', color='ABSINTH', gender='WOMAN', modele='BADEN', season='WINTER', material='100% SILK', description='<p>- Straight cut<br>- Long and wide sleeves<br>- Mandarin collar with buttons<br>- Button front opening<br>- 18mm Washed Silk</p>')

### mapping

#### `catalog`

In [60]:
types = set[str](item.type for item in items if item.type is not None)
len(types)

60

In [61]:
mappings = load_json("mapping/catalog_mappings.json")

In [62]:
for key, mapping in mappings.items():
    print(key, len(mapping))

claude 60
gemini 60
gpt 60


In [63]:
catalog_mapping = load_json("mapping/catalog_mapping.json")

In [64]:
if not catalog_mapping:
    catalog_mapping = defaultdict(set)

    for model, mapping in mappings.items():
        for item_type, catalog_ids in mapping.items():
            for catalog_id in catalog_ids:
                catalog_mapping[item_type].add(int(catalog_id))

    catalog_mapping = dict(catalog_mapping)
    for item_type, catalog_ids in catalog_mapping.items():
        catalog_mapping[item_type] = list(catalog_ids)

    save_json(catalog_mapping, "mapping/catalog_mapping.json")

#### `color`

In [65]:
colors = set[str](item.color for item in items if item.color is not None)
len(colors)

371

#### `material`

In [66]:
materials = set[str](item.material for item in items if item.material is not None)
len(materials)

188

## search

In [67]:
proxy_config = ProxyConfig()
client = VintedClient(proxy_config)
client.fetch_cookies()

In [ ]:
json_data = load_json(f"{DATA_DIR}/dataset.json") or []
dataset = Dataset.from_json(json_data)
index = load_json(f"{DATA_DIR}/index.json") or []

len(dataset), len(index)

(661, 1369)

In [ ]:
random.shuffle(items)
loop = tqdm(iterable=items, total=len(items))
n, n_success, errors = 0, 0, []

for item in loop:
    if item.id in index:
        continue 

    index.append(item.id)
    save_json(index, f"{DATA_DIR}/index.json")
    n += 1 

    if not item.modele or not item.type:
        continue

    search_params = VintedSearchParams(
        brand_ids=[VINTED_BRAND_ID], 
        query=item.modele.lower(), 
        catalog_ids=catalog_mapping[item.type],
        sort_option="relevance"
    )

    response = client.search(**search_params.model_dump())

    if not response.ok:
        errors.append(response.error)
        save_json(errors, f"{DATA_DIR}/errors.json")
        loop.set_description(f"status: {response.status_code}")
        continue

    if len(response.items) == 0:
        continue

    vinted_items = []
    
    for item_json in response.items:
        try:
            vinted_item = convert_to_vinted_item(item_json)
            vinted_items.append(vinted_item)
        
        except Exception as e:
            loop.set_description(f"error: {e}")
    
    if len(vinted_items) == 0:
        continue

    dataset.add(item, vinted_items)
    save_json(dataset.to_json(), f"{DATA_DIR}/dataset.json")

    n_success += 1
    success_rate = n_success / n

    loop.set_description(
        f"{n=}, {n_success=}, {success_rate=:.2%}"
    )

status: 500: 100%|██████████| 1369/1369 [11:17<00:00,  2.02it/s]                             


## postprocessing

In [133]:
json_data = load_json(f"{DATA_DIR}/dataset.json")
dataset = Dataset.from_json(json_data)
len(dataset)

661

In [134]:
num_vinted_items = [len(entry.vinted_items) for entry in dataset]
num_vinted_items_stats = get_statistics(values=num_vinted_items, quantiles_n=10) 
num_vinted_items_stats

{'average': 6.792738275340393,
 'min': 1,
 'max': 96,
 'q1': 1.0,
 'q2': 1.0,
 'q3': 2.0,
 'q4': 2.0,
 'q5': 3.0,
 'q6': 4.0,
 'q7': 5.0,
 'q8': 9.0,
 'q9': 14.0}

In [135]:
COMPARABLES_COUNT_BUCKETS = {
    3: ">=3",
    5: ">=5",
    9: ">=9",
    -1: "<3"
}

In [136]:
vinted_comparables_entries = [] 
run_id = str(uuid4())
created_at = datetime.now()

for entry in dataset:
    entries = build_vinted_comparables_entries(
        dataset_entry=entry,
        comparables_count_bucket_mapping=COMPARABLES_COUNT_BUCKETS,
        run_id=run_id,
        created_at=created_at,
        as_json=True,
    )

    vinted_comparables_entries.extend(entries)

In [137]:
save_json_lines(
    data=vinted_comparables_entries, 
    path=f"{DATA_DIR}/vinted_comparables.jsonl"
)